## 1. Imports

In [2]:
# !pip install scikit-survival pandas numpy matplotlib seaborn scipy

import numpy as np
import pandas as pd
import re
import parsing as ps
from collections import defaultdict
from scipy.stats import rankdata

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectFromModel

# Survival Analysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

print("Libraries Imported.")

Libraries Imported.


## 2. Load and clean data

In [3]:
# Load Datasets
# Adjust these paths if your data is in a different folder
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

# Clean Target
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Merge target into training data
df = df.merge(target_df[['ID', 'OS_YEARS', 'OS_STATUS']], on='ID', how='inner')

print(f"Training Data: {df.shape}")
print(f"Molecular Data: {maf_df.shape}")

Training Data: (3173, 11)
Molecular Data: (10935, 11)


## 3. Feature Engineering

In [4]:
# Cell 4 & 5: Optimized Feature Engineering (No Fragmentation Warning)

total_features = []

# --- 1. Cytogenetics (Fixed with concat) ---
print("Processing Cytogenetics...")

# Calculate Nmut
nmut_train = maf_df.groupby('ID').size().reset_index(name='Nmut')
nmut_test = maf_eval.groupby('ID').size().reset_index(name='Nmut')

df = df.merge(nmut_train, on='ID', how='left').fillna({'Nmut': 0})
df_eval = df_eval.merge(nmut_test, on='ID', how='left').fillna({'Nmut': 0})
total_features.append('Nmut')

# Parse ISCN and Concat
def process_cyto(dataset):
    cyto_dicts = dataset['CYTOGENETICS'].apply(ps.parse_CYTO)
    cyto_df = pd.DataFrame(cyto_dicts.tolist(), index=dataset.index).fillna(0)
    # Use concat instead of assignment to prevent fragmentation
    return pd.concat([dataset, cyto_df], axis=1)

df = process_cyto(df)
df_eval = process_cyto(df_eval)

# Add valid cyto columns to feature list
cyto_cols = [c for c in df.columns if any(x in c for x in ['+', '-', 't(', 'inv(', 'Complex', 'normal'])]
total_features.extend(cyto_cols)

# --- 2. Molecular Features (Fixed with concat) ---
print("Processing Molecular Features...")

# Clean Genes
maf_df['GENE_CLEAN'] = maf_df['GENE'].apply(ps.parse_GENE)
maf_eval['GENE_CLEAN'] = maf_eval['GENE'].apply(ps.parse_GENE)

# Pivot (VAF Weighting)
gene_train = maf_df.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
gene_test = maf_eval.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)

# Prefix
gene_train.columns = [f"GENE_{c}" for c in gene_train.columns]
gene_test.columns = [f"GENE_{c}" for c in gene_test.columns]

# Align Test to Train
gene_test = gene_test.reindex(columns=gene_train.columns, fill_value=0)

# Merge using merge (safe)
df = df.merge(gene_train, on='ID', how='left').fillna(0)
df_eval = df_eval.merge(gene_test, on='ID', how='left').fillna(0)
total_features.extend(gene_train.columns.tolist())

# One-Hot Encoding for Protein & Effect
for feature in ['EFFECT', 'PROTEIN_CHANGE']:
    # Parse
    col_name = 'temp'
    if feature == 'PROTEIN_CHANGE':
        maf_df[col_name] = maf_df[feature].apply(ps.parse_PROTEIN_CHANGE)
        maf_eval[col_name] = maf_eval[feature].apply(ps.parse_PROTEIN_CHANGE)
    else:
        maf_df[col_name] = maf_df[feature]
        maf_eval[col_name] = maf_eval[feature]
        
    # Dummies
    dummies_train = pd.get_dummies(maf_df[col_name], prefix=feature)
    dummies_train['ID'] = maf_df['ID']
    agg_train = dummies_train.groupby('ID').max()
    
    dummies_test = pd.get_dummies(maf_eval[col_name], prefix=feature)
    dummies_test['ID'] = maf_eval['ID']
    agg_test = dummies_test.groupby('ID').max()
    
    # Align
    agg_test = agg_test.reindex(columns=agg_train.columns, fill_value=0)
    
    # Merge
    df = df.merge(agg_train, on='ID', how='left').fillna(0)
    df_eval = df_eval.merge(agg_test, on='ID', how='left').fillna(0)
    total_features.extend(agg_train.columns.tolist())

# --- 3. Clinical & Final Cleanup ---
print("Processing Clinical Data...")
clinical_cols = ['BM_BLAST', 'WBC', 'ANC', 'PLT', 'HB', 'MONOCYTES']
for col in clinical_cols:
    df[col] = np.log1p(df[col])
    df_eval[col] = np.log1p(df_eval[col])

# Impute
imputer = SimpleImputer(strategy='median')
df[clinical_cols] = imputer.fit_transform(df[clinical_cols])
df_eval[clinical_cols] = imputer.transform(df_eval[clinical_cols])
total_features.extend(clinical_cols)

# Add Interactions
def add_interactions(data):
    if 'GENE_NPM1' in data.columns and 'GENE_FLT3' in data.columns:
        data['INT_NPM1_pos_FLT3_neg'] = ((data['GENE_NPM1'] > 0) & (data['GENE_FLT3'] == 0)).astype(int)
    if 'GENE_TP53' in data.columns and 'Complex_Karyotype' in data.columns:
        data['INT_TP53_Complex'] = ((data['GENE_TP53'] > 0) & (data['Complex_Karyotype'] > 0)).astype(int)
    return data

df = add_interactions(df)
df_eval = add_interactions(df_eval)

# Add interactions to feature list
for feat in ['INT_NPM1_pos_FLT3_neg', 'INT_TP53_Complex']:
    if feat in df.columns:
        total_features.append(feat)

# De-fragment frames before training (Important!)
df = df.copy()
df_eval = df_eval.copy()

# Final Feature List Cleanup
total_features = list(set([f for f in total_features if f in df.columns]))
print(f"Feature Engineering Complete. Total Features: {len(total_features)}")

Processing Cytogenetics...
Processing Molecular Features...
Processing Clinical Data...
Feature Engineering Complete. Total Features: 1773


## 4. Feature Selection

In [5]:
# Cell 5: Feature Selection

# Prepare X and y
X = df[total_features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

print(f"Starting feature selection on {X.shape[1]} features...")

# 1. Fit Cox Lasso
# l1_ratio=1.0 is Lasso (pure feature selection)
# We assume data is scaled or we use normalize=True if not in pipeline (CoxNet handles scale internally usually, but robust scaling helps)
# Here we fit directly on X.
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01, max_iter=100000)
cox_lasso.fit(X, y)

# 2. Extract Coefficients Manually
# coef_ is shape (n_features, n_alphas). We want the solution for our chosen alpha_min.
# We pick the last column (-1) which corresponds to alpha_min_ratio (0.01), 
# representing the most complex model allowed by our constraint.
coefficients = cox_lasso.coef_[:, -1]

# 3. Filter Features
# Identify which features have non-zero weights
non_zero_mask = coefficients != 0
selected_features = X.columns[non_zero_mask].tolist()

print(f"Selected {len(selected_features)} features out of {X.shape[1]}.")
print("First 10 selected features:", selected_features[:10])

# 4. Update X to use only selected features
X_selected = X[selected_features]

Starting feature selection on 1773 features...
Selected 22 features out of 1773.
First 10 selected features: ['GENE_ASXL1', 'GENE_EZH2', 'PROTEIN_CHANGE_0', 'EFFECT_stop_gained', 'HB', 'EFFECT_frameshift_variant', 'Complex_Karyotype', 'PROTEIN_CHANGE_34', 'INT_TP53_Complex', 'PLT']


## 5. Train RSF

In [6]:
print("Training Random Survival Forest on selected features...")

# Using aggressive parameters based on your "low min_samples_leaf" finding
rsf = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_leaf=10,      # Low leaf size for high detail
    max_features="sqrt",
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_selected, y)

# Quick check on Training accuracy
print(f"RSF Train Score: {rsf.score(X_selected, y):.4f}")

Training Random Survival Forest on selected features...
RSF Train Score: 0.7896


## 6. Train GBSA

In [7]:
print("Training Gradient Boosting...")

gbsa = GradientBoostingSurvivalAnalysis(
    n_estimators=300,
    learning_rate=0.05,      # Slower learning rate for better generalization
    max_depth=4,
    subsample=0.8,
    random_state=42
)

gbsa.fit(X_selected, y)

print(f"GBSA Train Score: {gbsa.score(X_selected, y):.4f}")

Training Gradient Boosting...


GBSA Train Score: 0.7949


## 7. Ensemble prediction

In [8]:
# 1. Prepare Evaluation Data (Align columns to Selected Features)
# This fixes the KeyError and ensures we only use the selected genes
X_eval_final = df_eval.reindex(columns=selected_features, fill_value=0)

# 2. Get Predictions
print("Predicting...")
pred_rsf = rsf.predict(X_eval_final)
pred_gbsa = gbsa.predict(X_eval_final)

# 3. Rank Averaging
# Convert risk scores to ranks (0 to 1) to make them comparable
rank_rsf = rankdata(pred_rsf)
rank_gbsa = rankdata(pred_gbsa)

# Weighted Blend (Forest is usually stronger, so 60% weight)
final_ensemble = (0.6 * rank_rsf) + (0.4 * rank_gbsa)

# 4. Save
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': final_ensemble
})

filename = "submission/submission_ensemble_gbm.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename}")
submission.head()

Predicting...
Saved submission/submission_ensemble_gbm.csv


,ID,risk_score
0,KYW1,864.4
1,KYW2,901.8
2,KYW3,387.2
3,KYW4,740.8
4,KYW5,750.4


In [9]:
# Cell 18: Validation of the Weighted Rank Ensemble (0.6*RSF + 0.4*GBSA)

from sklearn.model_selection import train_test_split
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
from sksurv.metrics import concordance_index_ipcw
from scipy.stats import rankdata

print("Running Ensemble Validation (85% Train / 15% Hold-out)...")

# 1. Split Data (Use X_selected from Cell 5 to save time/memory)
# Note: Ensure X_selected and y are available from previous cells
X_train_val, X_val, y_train_val, y_val = train_test_split(
    X_selected, y, test_size=0.15, random_state=42
)

# 2. Define Models with the settings we want to test
# RSF (Using the optimized parameters you found earlier)
rsf_val = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_leaf=10,      # Your tuned value
    max_depth=20,             # Your tuned value
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

# GBSA (Gradient Boosting)
gbsa_val = GradientBoostingSurvivalAnalysis(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

# 3. Train Models on the 85% Split
print("Training RSF...")
rsf_val.fit(X_train_val, y_train_val)

print("Training GBSA...")
gbsa_val.fit(X_train_val, y_train_val)

# 4. Predict on the 15% Validation Set
pred_rsf = rsf_val.predict(X_val)
pred_gbsa = gbsa_val.predict(X_val)

# 5. Apply Rank Blending (0.6 * RSF + 0.4 * GBSA)
rank_rsf = rankdata(pred_rsf)
rank_gbsa = rankdata(pred_gbsa)

ensemble_val_pred = (0.6 * rank_rsf) + (0.4 * rank_gbsa)

# 6. Evaluate C-Index for all three
c_index_rsf = concordance_index_ipcw(y_train_val, y_val, pred_rsf, tau=7)[0]
c_index_gbsa = concordance_index_ipcw(y_train_val, y_val, pred_gbsa, tau=7)[0]
c_index_ensemble = concordance_index_ipcw(y_train_val, y_val, ensemble_val_pred, tau=7)[0]

print("-" * 40)
print(f"Validation Results (tau=7):")
print(f"RSF Only:       {c_index_rsf:.4f}")
print(f"GBSA Only:      {c_index_gbsa:.4f}")
print(f"Ensemble (6:4): {c_index_ensemble:.4f}")
print("-" * 40)

if c_index_ensemble > max(c_index_rsf, c_index_gbsa):
    print("SUCCESS: Ensemble outperforms single models! -> Use submission_ensemble_final.csv")
else:
    winner = 'RSF' if c_index_rsf > c_index_gbsa else 'GBSA'
    print(f"RESULT: Single best model is {winner}. -> Use submission_rsf_optimized.csv (or GBSA equivalent)")

Running Ensemble Validation (85% Train / 15% Hold-out)...
Training RSF...
Training GBSA...
----------------------------------------
Validation Results (tau=7):
RSF Only:       0.7213
GBSA Only:      0.7198
Ensemble (6:4): 0.7213
----------------------------------------
RESULT: Single best model is RSF. -> Use submission_rsf_optimized.csv (or GBSA equivalent)
